# Module 4 - RAG pipeline

## Step 1 : Data Prepration & EDA

In [ ]:
from datasets import load_dataset
import pandas as pd 
import numpy as np
import re
import os
import joblib
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.tokenize import sent_tokenize
import torch

In [ ]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Amod/mental_health_counseling_conversations")
df = pd.DataFrame(ds['train'])
df.head()

In [ ]:
print("Before cleaning:\n")
# checking the number of Context & Response
print(f'Total Context & Response: {len(df)}')
print("="*50)

# checking the null values
print(f"Total null values: \n{df.isnull().sum()}")
print("="*50)

# checking duplicates
print(f"Duplicates rows: {df.duplicated().sum()}")
print(f"Duplicates Response: {df.duplicated(subset='Response').sum()}")
print(f"Duplicates Context: {df.duplicated(subset='Context').sum()}")
print("="*50)

In [ ]:
def clean_text(text):
    '''Clean text by removing URLs, HTML tags, extra spaces and newlines.'''
    text = str(text)

    # Remove HTML comments
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)
    # Remove quoted attribute values:  src="..."  or  src='...'  (with or without closing quote)
    text = re.sub(r'\w+\s*=\s*["\'][^"\']*["\']?', '', text)
    # Remove unquoted attribute values:  src=...
    text = re.sub(r'\w+\s*=\s*[^\s>"\']+', '', text)
    # Remove remaining tag shell  <...>  or  <...  (no closing >)
    text = re.sub(r'<[^>]*>?', '', text)

    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    text = re.sub(r'\b\S+\.(com|org|net|ca|io|pdf|html|htm)\S*', '', text)

    # Clean whitespace
    text = text.replace('\n', ' ')
    return re.sub(r'\s+', ' ', text).strip()

df['Context'] = df['Context'].apply(clean_text)
df['Response'] = df['Response'].apply(clean_text)

# Delete rows with empty Context or Response, drop short rows and duplicates
df = df[(df['Context'] != "") & (df['Response'] != "")]
df = df[df['Context'].str.len() >= 15]
df = df[df['Response'].str.len() >= 15]
df = df.drop_duplicates()

In [ ]:
print("After cleaning:\n")

# checking the number of Context & Response
print(f'Total Context & Response: {len(df)}')
print("="*50)

# checking the null values
print(f"Total null values: \n{df.isnull().sum()}")
print("="*50)

# checking duplicates
print(f"Duplicates rows: {df.duplicated().sum()}")
print(f"Duplicates Response: {df.duplicated(subset='Response').sum()}")
print(f"Duplicates Context: {df.duplicated(subset='Context').sum()}")
print("="*50)

In [ ]:
# There are some 'Contexts' that have multiple 'Responses'.
dup_context = df.groupby('Context')['Response'].nunique().sort_values(ascending=False)
dup_context[dup_context > 1]

In [ ]:
conflicts = (
    df.groupby('Context')
      .filter(lambda x: x['Response'].nunique() > 1)
      .sort_values('Context')
)

with open("conflicts_output.txt", "w", encoding="utf-8") as f:
    for context, group in conflicts.groupby('Context'):
        f.write("=" * 80 + "\n")
        f.write("CONTEXT:\n\n")
        f.write(str(context) + "\n")

        f.write("\nRESPONSES:\n\n")
        for i, response in enumerate(group['Response'].unique(), 1):
            f.write(f"{i}. {response}\n\n")

print("Saved to conflicts_output.txt")

In [ ]:
# saving the cleaned dataframe to a csv file in data folder
df.to_csv('data/df_cleaned.csv', index=False)

### Handle mutiple Response of the same Context

In [ ]:
# Load embedding model
model = SentenceTransformer('all-mpnet-base-v2')

# Precompute embeddings for all unique responses and cache to disk
emb_cache_path = os.path.join('data', 'response_embeddings.joblib')
responses = df['Response'].unique().tolist()

if os.path.exists(emb_cache_path):
    emb_lookup = joblib.load(emb_cache_path)
    print(f'Loaded cached embeddings for {len(emb_lookup)} responses')
else:
    print(f'Computing embeddings for {len(responses)} unique responses...')
    emb_array = model.encode(
        responses,
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=64,
        show_progress_bar=True
    )
    emb_lookup = {r: emb_array[i] for i, r in enumerate(responses)}
    os.makedirs(os.path.dirname(emb_cache_path), exist_ok=True)
    joblib.dump(emb_lookup, emb_cache_path)
    print(f'Saved embeddings cache to {emb_cache_path}')

#### Clustering-based Summarization

In [ ]:
def semantic_clustering_summarize(responses, emb_lookup, similarity_threshold=0.75):
    """ Cluster similar responses and return a representative response for each cluster.
    Args:
        responses (list): List of responses to cluster.
        emb_lookup (dict): Mapping response -> embedding (numpy array).
        similarity_threshold (float): Threshold for cosine similarity to consider responses as similar.
    Returns:
        list: List of representative responses for each cluster.
    """
    # if there is only one response
    if len(responses) <= 1:
        return responses

    # Build embeddings matrix from lookup; fall back to encoding missing responses
    missing = [r for r in responses if r not in emb_lookup]
    if missing:
        # encode missing ones on-the-fly
        missing_embs = model.encode(missing, convert_to_numpy=True, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
        for i, r in enumerate(missing):
            emb_lookup[r] = missing_embs[i]

    embeddings = np.vstack([emb_lookup[r] for r in responses])

    # Clustering
    clustering_model = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average', distance_threshold=1 - similarity_threshold)
    labels = clustering_model.fit_predict(embeddings)

    # Pick representative response
    unique_views = []

    for cluster_id in np.unique(labels):
        cluster_indices = np.where(labels == cluster_id)[0]
        cluster_embeddings = embeddings[cluster_indices]

        # centroid of cluster
        centroid = cluster_embeddings.mean(axis=0)
        # normalize centroid to unit length (avoid zero division)
        norm = np.linalg.norm(centroid)
        if norm > 0:
            centroid = centroid / (norm + 1e-12)

        # similarity to centroid
        similarities = cosine_similarity(cluster_embeddings, centroid.reshape(1, -1)).flatten()

        # best representative
        best_local_idx = np.argmax(similarities)
        best_idx = cluster_indices[best_local_idx]
        unique_views.append(responses[best_idx])

    return unique_views

In [ ]:
final_data = []

# Process each Context (with progress bar)
for context, group in tqdm(df.groupby('Context'), total=df['Context'].nunique()):
    raw_responses = group['Response'].unique().tolist()
    summarized_views = semantic_clustering_summarize(raw_responses, emb_lookup, similarity_threshold=0.75)

    # Guard against empty summarization
    if not summarized_views:
        summarized_views = raw_responses[:1]

    # Format responses
    formatted_responses = (
        "\n".join([f"• {r}" for r in summarized_views])
        if len(summarized_views) > 1
        else summarized_views[0])

    final_data.append({'Context': context, 'Responses': formatted_responses})

# Save final dataframe
summarized_df = pd.DataFrame(final_data)
out_path = os.path.join('data', 'semantic_clustered_rag.csv')
summarized_df.to_csv(out_path, index=False)
print(f'Saved summarized dataframe to {out_path}')
summarized_df.head()

## Step 2 : Chunking

In [ ]:
# Chunking Config 
CHUNK_SIZE_CHARS = 1500   # ~375 tokens per chunk
OVERLAP_CHARS    = 150    # chars carried over between chunks
BATCH_SIZE       = 64
MODEL_NAME       = 'all-mpnet-base-v2'

OUT_CHUNKS = os.path.join('data', 'chunks.parquet')
OUT_EMB    = os.path.join('artifacts', 'chunk_embeddings.joblib')
OUT_META   = os.path.join('artifacts', 'index_metadata.joblib')

print('Chunking config ready.')
print(f'  Chunk size : {CHUNK_SIZE_CHARS} chars')
print(f'  Overlap    : {OVERLAP_CHARS} chars')
print(f'  Model      : {MODEL_NAME}')

In [ ]:
def split_bullets(text):
    """
    Split a Responses field into individual therapist answers.
    Handles bullet markers: •  -  *  1.  and newlines.
    """
    if not text:
        return []
    parts = re.split(r'\n+', str(text))
    bullets = []
    for p in parts:
        s = p.strip()
        if not s:
            continue
        s = re.sub(r'^\s*(?:•|\-|\*|\d+\.)\s*', '', s)
        s = s.strip()
        if s:
            bullets.append(s)
    return bullets


def chunk_text(text, size=1500, overlap=150):
    text = str(text).strip()
    if not text:
        return []
    if len(text) <= size:
        return [text]
    
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        # Hard cap: if single sentence exceeds size, split it
        if len(sentence) > size:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            overlap_text = sentence[-overlap:] if overlap > 0 else ""
            current_chunk = overlap_text
            for i in range(0, len(sentence), size):
                chunks.append(sentence[i:i+size])
            current_chunk = ""
        elif len(current_chunk) + len(sentence) + 1 > size:
            chunks.append(current_chunk.strip())
            overlap_text = current_chunk[-overlap:] if overlap > 0 else ""
            current_chunk = overlap_text + " " + sentence
        else:
            current_chunk += " " + sentence
    
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks


In [ ]:
# Load the semantic-clustered output from Step 1
chunking_df = pd.read_csv(os.path.join('data', 'semantic_clustered_rag.csv'))
print(f'Loaded {len(chunking_df):,} rows')
print(f'Columns: {chunking_df.columns.tolist()}')
chunking_df.head(3)

In [ ]:
nltk.download('punkt_tab')

In [ ]:
records = []

for i, row in tqdm(chunking_df.iterrows(), total=len(chunking_df), desc='Chunking rows'):
    question = str(row.get('Context', row.iloc[0])).strip()

    # Support both 'Responses' and 'Response' column names
    responses_field = row.get('Responses') if 'Responses' in chunking_df.columns else row.get('Response')

    bullets = split_bullets(responses_field) if responses_field else []
    if not bullets:
        bullets = [responses_field] if responses_field else [question]

    for bi, bullet in enumerate(bullets):
        qa_text = f'Q: {question}\nA: {bullet}'
        for ci, chunk in enumerate(chunk_text(qa_text, size=CHUNK_SIZE_CHARS, overlap=OVERLAP_CHARS)):
            records.append({
                'context_id'       : int(i),
                'bullet_index'     : int(bi),
                'chunk_index'      : int(ci),
                'original_response': bullet,
                'text'             : chunk,
            })

chunks_df = pd.DataFrame(records)
print(f'Original rows : {len(chunking_df):,}')
print(f'Total chunks  : {len(chunks_df):,}')
chunks_df.head(3)

In [ ]:
chunks_df['char_len'] = chunks_df['text'].str.len()

print('Chunk character length stats:')
print(chunks_df['char_len'].describe().round(1))
print()
print(f'Chunks under 500 chars  : {(chunks_df["char_len"] < 500).sum():,}')
print(f'Chunks 500–1500 chars   : {((chunks_df["char_len"] >= 500) & (chunks_df["char_len"] <= 1500)).sum():,}')
print(f'Chunks over 1500 chars  : {(chunks_df["char_len"] > 1500).sum():,}')

In [ ]:
# Save chunks to Parquet for efficient storage and later embedding
os.makedirs('data', exist_ok=True)
chunks_df.to_parquet(OUT_CHUNKS, index=False)
print(f'Saved {len(chunks_df):,} chunks → {OUT_CHUNKS}')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading embedding model: {MODEL_NAME} on {device}...')
embed_model = SentenceTransformer(MODEL_NAME, device=device)

texts    = chunks_df['text'].tolist()
emb_list = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Embedding batches'):
    batch = texts[i : i + BATCH_SIZE]
    emb   = embed_model.encode(
        batch,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    emb_list.append(emb)

embeddings = np.vstack(emb_list)
print(f'Embeddings shape: {embeddings.shape}')  # (n_chunks, 384)

In [ ]:
joblib.dump(embeddings,  OUT_EMB)
joblib.dump(chunks_df.to_dict(orient='records'), OUT_META)

print(f'Saved embeddings → {OUT_EMB}   shape={embeddings.shape}')
print(f'Saved metadata   → {OUT_META}  records={len(chunks_df):,}')

In [ ]:
# Quick check: retrieve nearest chunk to a sample query
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

sample_query = 'I feel hopeless and do not know what to do'
q_emb = embed_model.encode([sample_query], normalize_embeddings=True)
scores = cos_sim(q_emb, embeddings)[0]
top_idx = scores.argsort()[::-1][:3]

print(f'Query: "{sample_query}"\n')
for rank, idx in enumerate(top_idx, 1):
    print(f'--- Rank {rank}  (score={scores[idx]:.4f}) ---')
    print(chunks_df.iloc[idx]['text'][:300])
    print()

## Step 3 : Store in VectorDB

In [ ]:
from qdrant_client import QdrantClient
from dotenv import load_dotenv
load_dotenv()

# Initialize Qdrant client using environment variables for URL and API key
qdrant_client = QdrantClient(
    url     = os.getenv("QDRANT_URL"),
    api_key = os.getenv("QDRANT_API_KEY"),
)

In [ ]:
# Recreate collection with appropriate vector size and distance metric
from qdrant_client.http.models import Distance, VectorParams
qdrant_client.recreate_collection(
    collection_name="mental_health_chunks",
    vectors_config=VectorParams(
        size=768,
        distance=Distance.COSINE,
    ),
)

In [ ]:
from qdrant_client.http.models import PointStruct
import time

# Load your saved embeddings and chunks
embeddings = joblib.load('artifacts/chunk_embeddings.joblib')
metadata   = joblib.load('artifacts/index_metadata.joblib')

# Prepare points for Qdrant upload
points = [
    PointStruct(
        id=idx,
        vector=embeddings[idx].astype('float32').tolist(),
        payload={
            "context_id"        : rec["context_id"],
            "bullet_index"      : rec["bullet_index"],
            "chunk_index"       : rec["chunk_index"],
            "original_response" : rec["original_response"],
            "text"              : rec["text"],
        }
    )
    for idx, rec in enumerate(metadata)
]

# upload points in batches with retry logic 
# (if a batch fails, it will retry up to 3 times with a delay)

BATCH = 32
for i in tqdm(range(0, len(points), BATCH), desc='Uploading to Qdrant'):
    batch = points[i : i + BATCH]
    
    for attempt in range(3):
        try:
            qdrant_client.upsert(
                collection_name = "mental_health_chunks", 
                points = batch,
                wait = False)
            print(f"Uploaded {min(i+BATCH, len(points))}/{len(points)}")
            break
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2)

print("All points uploaded")

In [ ]:
# Verify count in Qdrant matches number of points uploaded
info = qdrant_client.get_collection("mental_health_chunks")
print(f'Vectors in Qdrant : {info.points_count:,}')
print(f'Expected          : {len(points):,}')
assert info.points_count == len(points), 'Mismatch! Some points may not have uploaded.'

In [ ]:
# ------------------------------------------------------------------
# Quick check: retrieve nearest chunk to a query using Qdrant search
# ------------------------------------------------------------------

# Reuse embed_model if already loaded, otherwise reload
if 'embed_model' not in dir():
    embed_model = SentenceTransformer(MODEL_NAME)

sample_query = 'I feel hopeless and do not know what to do'

q_emb = embed_model.encode(
    [sample_query],
    normalize_embeddings=True
)[0].tolist()

results = qdrant_client.query_points(
    collection_name="mental_health_chunks",
    query=q_emb,
    limit=3,
    with_payload=True,
)

print(f'Query: "{sample_query}"\n')

for rank, r in enumerate(results.points, 1):
    print(f'--- Rank {rank} (score={r.score:.4f}) ---')
    print(r.payload['text'][:300])
    print()

In [ ]:
from rank_bm25 import BM25Okapi

# Load chunk texts (already saved in Step 2)
# Position in this list == Qdrant point id  (because you uploaded with id=idx)
bm25_texts       = pd.read_parquet(os.path.join('data', 'chunks.parquet'))['text'].tolist()
tokenized_corpus = [text.lower().split() for text in bm25_texts]
bm25_index       = BM25Okapi(tokenized_corpus)

print(f"BM25 index built on {len(bm25_texts):,} documents")

## Step 4 : Hybrid Retrieval (Semantic 0.7 + BM25 0.3)

In [ ]:
SEMANTIC_WEIGHT = 0.7
BM25_WEIGHT     = 0.3
CANDIDATE_POOL  = 50   # how many candidates to pull from each method before re-ranking


def _normalize(score_dict):
    """Min-max normalize a {id: score} dict to [0, 1]."""
    if not score_dict:
        return {}
    values = list(score_dict.values())
    lo, hi = min(values), max(values)
    if hi == lo:
        return {k: 1.0 for k in score_dict}
    return {k: (v - lo) / (hi - lo) for k, v in score_dict.items()}


def retrieve_chunks_hybrid(query_text, top_k=5):
    """
    Hybrid retrieval: 0.7 * semantic + 0.3 * BM25

    1. Semantic  → ask Qdrant for top CANDIDATE_POOL results + scores
    2. BM25      → score all docs locally, take top CANDIDATE_POOL
    3. Union     → merge both candidate sets
    4. Normalize → scale each score set to [0, 1] independently
                   (needed because cosine scores ~0.4-0.9 and BM25 scores ~0-30)
    5. Combine   → final = 0.7 * sem_norm + 0.3 * bm25_norm
    6. Return    → fetch payloads from Qdrant for the top_k winners
    """

    # ── 1. Semantic search ──────────────────────────────────────────────────
    q_emb = embed_model.encode([query_text], normalize_embeddings=True)[0].tolist()

    semantic_hits = qdrant_client.query_points(
        collection_name="mental_health_chunks",
        query=q_emb,
        limit=CANDIDATE_POOL,
        with_payload=True,
    )
    semantic_scores = {r.id: r.score for r in semantic_hits.points}

    # ── 2. BM25 search ──────────────────────────────────────────────────────
    tokenized_query = query_text.lower().split()
    bm25_all_scores = bm25_index.get_scores(tokenized_query)  # array of length n_chunks

    bm25_top_ids = np.argsort(bm25_all_scores)[::-1][:CANDIDATE_POOL]
    bm25_scores  = {int(i): float(bm25_all_scores[i]) for i in bm25_top_ids}

    # ── 3. Union of both candidate sets ────────────────────────────────────
    all_ids = set(semantic_scores.keys()) | set(bm25_scores.keys())

    # ── 4. Normalize ────────────────────────────────────────────────────────
    sem_norm  = _normalize(semantic_scores)
    bm25_norm = _normalize(bm25_scores)

    # ── 5. Combine ──────────────────────────────────────────────────────────
    combined = []
    for cid in all_ids:
        sem   = sem_norm.get(cid,  0.0)
        bm25  = bm25_norm.get(cid, 0.0)
        score = SEMANTIC_WEIGHT * sem + BM25_WEIGHT * bm25
        combined.append((cid, score))

    combined.sort(key=lambda x: x[1], reverse=True)
    top_ids = [cid for cid, _ in combined[:top_k]]

    # ── 6. Fetch payloads from Qdrant ───────────────────────────────────────
    fetched       = qdrant_client.retrieve(
        collection_name="mental_health_chunks",
        ids=top_ids,
        with_payload=True,
    )
    id_to_payload = {r.id: r.payload for r in fetched}

    # Return in ranked order, preserving the combined score ranking
    return [id_to_payload[cid]["text"] for cid in top_ids if cid in id_to_payload]

### Testing Retrieval

In [ ]:
query   = "I feel happy and do not know what to do"
results = retrieve_chunks_hybrid(query, top_k=3)

print(f'Query: "{query}"\n')
for i, text in enumerate(results, 1):
    print(f"--- Rank {i} ---")
    print(text[:300])
    print()

## Step 5 : Response

In [ ]:
from openai import OpenAI

client_llm = OpenAI(
    api_key  = os.getenv("OPENAI_API_KEY"),
    base_url = os.getenv("OPENAI_BASE_URL"),
)

MAIN_MODEL        = "llama-3.1-8b-instant"
TRANSLATOR_MODEL  = "llama-3.1-8b-instant"

### Language Detector Model

In [ ]:
# Map your language detector's 3-letter output codes to full language names
# The LLM understands "Arabic" much better than "ara" in the prompt
LANGUAGE_NAMES = {
    "pt": "Portuguese",
    "bg": "Bulgarian",
    "zh": "Chinese",
    "th": "Thai",
    "ru": "Russian",
    "pl": "Polish",
    "ur": "Urdu",
    "sw": "Swahili",
    "tr": "Turkish",
    "es": "Spanish",
    "ar": "Arabic",
    "it": "Italian",
    "hi": "Hindi",
    "de": "German",
    "el": "Greek",
    "nl": "Dutch",
    "fr": "French",
    "vi": "Vietnamese",
    "en": "English",
    "ja": "Japanese"
}

svc_model = joblib.load("models/language_svc_model.pkl")
vectorizer = joblib.load("models/tfidf_vectorizer.pkl")

# create a function to predict the language of a given text
def predict_language(text):
    text_vec = vectorizer.transform([text])
    prediction = svc_model.predict(text_vec)
    return LANGUAGE_NAMES[prediction[0]]

In [ ]:
def translate_to_english(text, source_language):
    """
    source_language is now a full name e.g. 'arabic', 'french'
    """
    response = client_llm.chat.completions.create(
        model    = TRANSLATOR_MODEL,
        messages = [
            {
                "role"   : "system",
                "content": (
                    f"You are a translation engine. "
                    f"Translate the following {source_language} text to English. "
                    f"Return ONLY the translated text — no explanation, "
                    f"no preamble, nothing else."
                )
            },
            {
                "role"   : "user",
                "content": text
            }
        ],
        max_tokens  = 512,
        temperature = 0.0,
    )
    return response.choices[0].message.content.strip()

### Emotional Model

In [ ]:
EMOTION_TONE = {
    "sadness" : "Be empathetic and gentle. Acknowledge their pain and show you understand before offering help.",
    "joy"     : "Be warm and encouraging. Celebrate their positive feelings while offering supportive guidance.",
    "fear"    : "Be calm and reassuring. Help ground them and reduce their anxiety with clear, steady guidance.",
    "anger"   : "Be calm and non-confrontational. Validate their frustration without escalating, and guide them gently.",
    "love"    : "Be warm and supportive. Acknowledge their feelings of connection and guide them with care.",
    "surprise": "Be clear and informative. Help them process the unexpected situation with steady, factual support.",
}

In [ ]:
def build_prompt(query, chunks, emotion, language):

    tone = EMOTION_TONE.get(emotion, "Be empathetic and supportive.")

    system_message = f"""You are a compassionate mental health support assistant.
Your role is to provide empathetic, grounded, and helpful responses based ONLY on the context provided.

Tone instruction: {tone}

Rules:
- Answer ONLY from the provided context. Do not invent information.
- If the context does not contain a relevant answer, say so honestly and suggest seeking professional help.
- Keep your response concise, warm, and easy to understand.
- You MUST respond in {language}.
"""

    context_block = "\n\n".join(
        [f"[Context {i+1}]:\n{chunk}" for i, chunk in enumerate(chunks)]
    )

    user_message = f"""Context from knowledge base:
{context_block}

User question: {query}
"""
    return system_message, user_message

In [ ]:
def call_llm(system_message, user_message, history=[]):
    
    messages = [{"role": "system", "content": system_message}]
    
    # inject previous conversation turns
    messages.extend(history)
    
    # add the new user message
    messages.append({"role": "user", "content": user_message})

    response = client_llm.chat.completions.create(
        model       = MAIN_MODEL,
        messages    = messages,
        max_tokens  = 1024,
        temperature = 0.7,
    )
    return response.choices[0].message.content.strip()

In [ ]:
def get_rag_answer(original_query, detected_language, emotion, history=[], top_k=5):

    if detected_language != "english":
        english_text = translate_to_english(original_query, detected_language)
    else:
        english_text = original_query

    chunks               = retrieve_chunks_hybrid(english_text, top_k=top_k)
    system_msg, user_msg = build_prompt(english_text, chunks, emotion, detected_language)
    answer               = call_llm(system_msg, user_msg, history)  # ← pass history

    return answer

In [ ]:
query = "انا مبسوط جدا و الدنيا زى الفل تقدر تقزلى اعمل اية عشان احافظ على طاقتى الايجابية دى "
answer = get_rag_answer(
    original_query    = query,
    detected_language = predict_language(query),
    top_k             = 5
)

print(answer)

## Intent Classifier

In [ ]:
import os
import json
from typing import Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_core.prompts import (
    PromptTemplate,
    FewShotPromptTemplate,
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_groq import ChatGroq   
import logging
import sys

load_dotenv()

In [ ]:
logging.basicConfig(
    level    = logging.INFO,
    format   = "%(asctime)s [%(levelname)s] %(message)s",
    datefmt  = "%H:%M:%S",
    handlers = [logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

In [ ]:
SYSTEM_PROMPT = """You are an advanced triage and routing assistant for a specialized mental health support system. 
Your sole responsibility is to analyze the user's input and categorize it into exactly ONE of the allowed intent classes.

CRITICAL INSTRUCTIONS:
1. Base your classification entirely on the core intent of the user's statement.
2. Rely heavily on the few-shot examples provided below to understand the classification boundaries.
3. Choose exactly one of the allowed categories. Do not invent new categories.

ALLOWED CATEGORIES AND DEFINITIONS:

- greeting: 
  The user is starting a conversation, saying hello, or checking if someone is online.
  (e.g., "Hi", "Hello", "Is anyone there?", "Good morning")

- goodbye: 
  The user is ending the conversation, signing off, or indicating they are leaving.
  (e.g., "Bye", "See you later", "Talk to you tomorrow", "I'm heading out")

- gratitude: 
  The user is thanking the system, expressing appreciation, or confirming that their issue was resolved.
  (e.g., "Thank you so much", "Thanks for the help", "That makes sense, thank you", "Appreciate it")

- asking_mental_health_question: 
  The user is actively seeking help, coping mechanisms, definitions, or advice regarding mental health conditions, emotions, symptoms, or psychological well-being. This is a critical class that triggers our clinical knowledge retrieval pipeline.
  (e.g., "How do I deal with panic attacks?", "I'm feeling incredibly anxious right now", "What are the signs of burnout?", "Can you give me tips for depression?")

- out_of_scope: 
  The user is asking about general knowledge, coding, math, recipes, casual chit-chat, or anything completely unrelated to mental health support.
  (e.g., "What is the capital of France?", "Write a python script", "Tell me a joke", "How's the weather?")

Analyze the context carefully. If a user says "Hello, I am having a panic attack", the primary intent is 'asking_mental_health_question', not 'greeting'. Prioritize clinical inquiries over conversational fluff.
"""

In [ ]:
class IntentResponse(BaseModel):
    intent: Literal[
        "greeting",
        "goodbye",
        "gratitude",
        "asking_mental_health_question",
        "out_of_scope"
    ] = Field(description="The classified intent of the user's message.")

In [ ]:
class IntentClassifier:

    def __init__(
        self,
        model_name  : str   = "llama-3.3-70b-versatile",  
        temperature : float = 0,
    ):
        logger.info(f"Initializing IntentClassifier with model='{model_name}', temp={temperature}")
        self.model_name  = model_name
        self.temperature = temperature
        self.examples    = self._load_examples()
        self.chain       = self._build_chain()
        logger.info("IntentClassifier initialization complete.")

    def _load_examples(self):
        logger.info("Attempting to load few-shot examples from 'intentExamples.json'...")
        try:
            with open("intentExamples.json", "r", encoding="utf-8") as f:
                examples = json.load(f)
            logger.info(f"Successfully loaded {len(examples)} few-shot examples.")
            return examples
        except Exception as e:
            logger.error(f"Failed to load examples file: {str(e)}")
            raise

    def _build_chain(self):
        logger.info("Assembling LangChain LCEL pipeline components...")

        example_prompt = PromptTemplate(
            input_variables = ["query", "intent"],
            template        = "User: {query}\nIntent: {intent}",
        )

        few_shot_prompt = FewShotPromptTemplate(
            examples        = self.examples,
            example_prompt  = example_prompt,
            prefix          = SYSTEM_PROMPT,
            suffix          = "User: {input}\nIntent:",
            input_variables = ["input"],
        )

        system_message_prompt = SystemMessagePromptTemplate(prompt=few_shot_prompt)
        human_message_prompt  = HumanMessagePromptTemplate.from_template("{input}")
        chat_prompt           = ChatPromptTemplate.from_messages(
            [system_message_prompt, human_message_prompt]
        )
        logger.info("Prompt templates structured successfully.")

        logger.info(f"Connecting to Groq API for model '{self.model_name}'...")  
        llm = ChatGroq(                                   
            model       = self.model_name,
            temperature = self.temperature,
            api_key     = os.getenv("OPENAI_API_KEY"),   
        )

        logger.info("Binding Pydantic output schema (IntentResponse) to LLM...")
        structured_llm = llm.with_structured_output(IntentResponse)

        chain = chat_prompt | structured_llm
        logger.info("LCEL routing chain compiled successfully.")
        return chain

    def predict(self, text: str) -> str:
        logger.info(f"Received user input for prediction: '{text}'")
        logger.info("Invoking LLM chain...")

        try:
            prediction: IntentResponse = self.chain.invoke({"input": text})
            logger.info(f"Extracted intent: '{prediction.intent}'")
            return prediction.intent
        except Exception as e:
            logger.error(f"Error during chain execution: {str(e)}")
            raise

In [ ]:
def get_intent_with_context(english_text, history):
    """
    If there is previous history, give the intent classifier
    the last user message as context so it understands follow-ups.
    """
    if history and len(history) >= 2:
        last_user_msg = history[-2]["content"]  # last user turn
        context_input = (
            f"Previous message: {last_user_msg}\n"
            f"Current message: {english_text}"
        )
    else:
        context_input = english_text

    return intent_classifier.predict(context_input)

In [ ]:
DIRECT_RESPONSE_PROMPTS = {
    "greeting"   : "The user is greeting you. Respond with a warm, friendly greeting and ask how you can help them today.",
    "goodbye"    : "The user is saying goodbye. Respond with a warm farewell and remind them you are here if they need support.",
    "gratitude"  : "The user is thanking you. Respond with a kind acknowledgment and let them know you are always here to help.",
    "out_of_scope": "The user is asking something outside your scope as a mental health assistant. Politely let them know you can only help with mental health related topics and invite them to ask something relevant.",
}
def get_direct_response(intent, detected_language):
    # No history passed here — these are simple standalone responses
    instruction    = DIRECT_RESPONSE_PROMPTS.get(intent, "Respond helpfully and kindly.")
    system_message = f"""You are a compassionate mental health support assistant.
{instruction}
You MUST respond in {detected_language}.
Keep your response short and natural.
"""
    response = client_llm.chat.completions.create(
        model    = MAIN_MODEL,
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user",   "content": ""},
        ],
        max_tokens  = 256,
        temperature = 0.7,
    )
    return response.choices[0].message.content.strip()

In [ ]:
def test_system():
    classifier = IntentClassifier()

    test_cases = {
        "Hey, how's it going?"                        : "greeting",
        "Goodbye, see you tomorrow."                  : "goodbye",
        "Thank you so much for the advice!"           : "gratitude",
        "I'm feeling deeply anxious and can't sleep." : "asking_mental_health_question",
        "Can you write a poem about space?"           : "out_of_scope",
    }

    passed = 0
    for text, expected in test_cases.items():
        result = classifier.predict(text)
        status = "PASS" if result == expected else f"FAIL (got '{result}', expected '{expected}')"
        print(f"{status}  |  '{text}'")
        if result == expected:
            passed += 1

    print(f"\n{passed}/{len(test_cases)} passed")

test_system()

## Integration

In [ ]:
# Module 3 - Intent Classifier
# intentExamples.json must be in the same folder as this notebook
intent_classifier = IntentClassifier()
print("Intent classifier ready")

In [ ]:
def pipeline(user_message, history=[]):

    detected_language = predict_language(user_message)

    if detected_language != "english":
        english_text = translate_to_english(user_message, detected_language)
    else:
        english_text = user_message

    emotion = "joy"

    # ── use context-aware intent detection ──────────────────────────────────
    intent = get_intent_with_context(english_text, history)  

    if intent == "asking_mental_health_question":
        response = get_rag_answer(user_message, detected_language, emotion, history, top_k=5)
    else:
        response = get_direct_response(intent, detected_language)  # ← no history

    history.append({"role": "user",      "content": user_message})
    history.append({"role": "assistant", "content": response})

    return {
        "response" : response,
        "intent"   : intent,
        "emotion"  : emotion,
        "language" : detected_language,
        "history"  : history,
    }

In [ ]:
# ── Test 1 : English mental health question → should trigger RAG ────────────
result = pipeline("I have been feeling very anxious and cannot sleep at night")
print(f"Language : {result['language']}")
print(f"Intent   : {result['intent']}")
# print(f"Emotion  : {result['emotion']}")
print(f"Response : {result['response']}")
print("=" * 60)

In [ ]:
# ── Test 2 : Arabic mental health question → translate then RAG ─────────────
result = pipeline("أشعر بالحزن الشديد ولا أعرف كيف أتعامل مع مشاعري")
print(f"Language : {result['language']}")
print(f"Intent   : {result['intent']}")
print(f"Emotion  : {result['emotion']}")
print(f"Response : {result['response']}")
print("=" * 60)

In [ ]:
# ── Test 3 : Greeting → should skip RAG entirely ────────────────────────────
result = pipeline("Hello, is anyone there?")
print(f"Language : {result['language']}")
print(f"Intent   : {result['intent']}")
print(f"Response : {result['response']}")
print("=" * 60)

In [ ]:
# ── Test 4 : Out of scope → should return polite refusal ────────────────────
result = pipeline("What is the capital of France?")
print(f"Language : {result['language']}")
print(f"Intent   : {result['intent']}")
print(f"Response : {result['response']}")
print("=" * 60)

In [ ]:
# Simulate a multi-turn conversation
history = []

# Turn 1
result  = pipeline("I feel very anxious and cannot sleep", history)
history = result["history"]
print(f"Turn 1 → {result['response']}\n")

# Turn 2 — LLM now knows context from Turn 1
result  = pipeline("What can I do about it?", history)
history = result["history"]
print(f"Turn 2 → {result['response']}\n")

# Turn 3
result  = pipeline("Thank you that really helps", history)
history = result["history"]
print(f"Turn 3 → {result['response']}\n")